In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.metrics import roc_curve, auc , confusion_matrix, accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split


from keras.models import load_model
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.optimizers import RMSprop


from dataset_preparation import awgn, LoadDataset, ChannelIndSpectrogram
from deep_learning_models import TripletNet, identity_loss

Using TensorFlow backend.
/opt/conda/envs/project/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:523: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
/opt/conda/envs/project/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:524: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
/opt/conda/envs/project/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:525: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
/opt/conda/envs/project/lib/python3.6/site-packages/t

In [2]:



#!unzip -q /workspaces/work/dataset_training_aug.zip
training_dataset_path = "/workspaces/work/Dataset/dataset_training_no_aug.h5"
#"/workspaces/work/Dataset/dataset_training_no_aug.h5"



In [3]:

file_path = training_dataset_path        
dev_range = np.arange(0,30, dtype = int), 
pkt_range = np.arange(0,15000, dtype = int), 
snr_range = np.arange(20,80)

LoadDatasetObj = LoadDataset()
    
# Load preamble IQ samples and labels.
data, labels = LoadDatasetObj.load_iq_samples(file_path, 
                                                 dev_range, 
                                                 pkt_range)

Dataset information: Dev 1 to Dev 30, 500 packets per device.


In [4]:



data = awgn(data, snr_range)

In [5]:
np.unique(labels)

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29])

In [ ]:
a = np.array([1,2,3,4])
# print(a[0:2])
# print(a[2:])

new_a = np.where(a==2, 'gotyou', a)
print(new_a)



In [7]:
data.shape


(10000, 8192)

In [ ]:
labels.shape

In [11]:
data.shape

(15000, 1, 8192)

In [7]:
labels.shape

(10000, 1)

In [6]:
#SOME PREPROCESSING
data = np.real(data).reshape(data.shape[0], 1, data.shape[1])
data = data/np.max(data)
data.shape
data_train, data_test, labels_train, labels_test = train_test_split(data, labels, test_size=0.2, shuffle=True)

# HYPERPAMETER TUNNING
# learning rate of optimiser, high epochs for low lr
# padding in layer 1 and corresponding maxpooling layers in cnn_model

# NOTES
# default adam with binary_crossentropy not good, but lr=0.0001
# 0.001 sgd with binary_crossentropy good
# default focalbinary_crossentropy with default adam good
# combimed cnn+lstm with 0.001 rms prop good 

In [12]:
# Assuming your data has shape (15000, 1, 8192)
# Reshape data to (15000, 8192)
data = data.reshape(data.shape[0], -1)

# Apply StandardScaler
scaler = StandardScaler()
data = scaler.fit_transform(data)

# After scaling, if you want to keep the shape (15000, 1, 8192)
# Reshape data back to (15000, 1, 8192)
data = data.reshape(data.shape[0], 1, -1)


In [13]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Step 1: Create Labels
num_legitimate_labels = 20
num_illegitimate_labels = 10
packets_per_label = 1000
total_samples = num_legitimate_labels * packets_per_label + num_illegitimate_labels * packets_per_label

legitimate_labels = np.zeros(num_legitimate_labels * packets_per_label, dtype=int)
illegitimate_labels = np.ones(num_illegitimate_labels * packets_per_label, dtype=int)
labels = np.concatenate([legitimate_labels, illegitimate_labels])

# Step 2: Preprocess Data
#data = data # Load your data
#scaler = StandardScaler()
#data = scaler.fit_transform(data)


# Step 3: Build CNN Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dropout, Flatten, Dense, BatchNormalization

cnn_model = Sequential()
cnn_model.add(Conv1D(filters=32, kernel_size=3, activation='relu', padding='same', input_shape=(1,8192)))
cnn_model.add(BatchNormalization())
cnn_model.add(MaxPooling1D(pool_size=2, padding='same'))
cnn_model.add(Dropout(0.3))
cnn_model.add(Conv1D(64, kernel_size=3, activation='relu', padding='same'))
cnn_model.add(BatchNormalization())
cnn_model.add(MaxPooling1D(pool_size=2, padding='same'))
cnn_model.add(Dropout(0.3))
cnn_model.add(Conv1D(128, kernel_size=3, activation='relu', padding='same'))
cnn_model.add(BatchNormalization())
cnn_model.add(MaxPooling1D(pool_size=2, padding='same'))
cnn_model.add(Dropout(0.3))
cnn_model.add(Flatten())
cnn_model.add(Dense(128, activation='relu'))
cnn_model.add(Dropout(0.5))
cnn_model.add(Dense(1, activation='sigmoid'))

cnn_model.compile(optimizer=Adam(lr=0.001), loss='binary_crossentropy', metrics=['accuracy'])


callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.0001),
   # EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
]

#
# Step 4: Train Model
#X_train, X_val, y_train, y_val = train_test_split(data, labels, test_size=0.2, random_state=42)

history = cnn_model.fit(data_train, labels_train, epochs=20, batch_size=32, validation_data=(data_test, labels_test), 
                   callbacks=[  ReduceLROnPlateau(factor=0.2, patience=2)]
                    )

# Step 5: Evaluate Model
labels_pred = model.predict(data_test)
labels_pred_classes = np.round(labels_pred)

accuracy = accuracy_score(labels_test, labels_pred_classes)
#precision = precision_score(y_val, y_pred_classes)
#recall = recall_score(y_val, y_pred_classes)

#print("Accuracy:", accuracy)
#print("Precision:", precision)
#print("Recall:", recall)


Train on 12000 samples, validate on 3000 samples
Epoch 1/20
12000/12000 [==============================] - 4s 341us/step - loss: -191.0531 - acc: 0.0339 - val_loss: -215.8333 - val_acc: 0.0310
Epoch 2/20
12000/12000 [==============================] - 3s 276us/step - loss: -215.0128 - acc: 0.0339 - val_loss: -215.8333 - val_acc: 0.0310
Epoch 3/20
12000/12000 [==============================] - 3s 292us/step - loss: -215.0564 - acc: 0.0339 - val_loss: -215.8333 - val_acc: 0.0310
Epoch 4/20
12000/12000 [==============================] - 3s 287us/step - loss: -215.0515 - acc: 0.0339 - val_loss: -215.8333 - val_acc: 0.0310
Epoch 5/20
12000/12000 [==============================] - 3s 289us/step - loss: -215.0520 - acc: 0.0339 - val_loss: -215.8333 - val_acc: 0.0310
Epoch 6/20
12000/12000 [==============================] - 3s 286us/step - loss: -215.0641 - acc: 0.0339 - val_loss: -215.8333 - val_acc: 0.0310
Epoch 7/20
12000/12000 [==============================] - 3s 274us/step - loss: -215.06

NameError: name 'model' is not defined

: 

In [21]:
import numpy as np

# Assuming you have 20 unique labels
num_legitimate_labels = 7
num_illegitimate_labels = 3
num_packets_per_label = 1000

# Create labels for legitimate data (0)
legitimate_labels = np.zeros(num_legitimate_labels * num_packets_per_label, dtype=int)

# Create labels for illegitimate data (1)
illegitimate_labels = np.ones(num_illegitimate_labels * num_packets_per_label, dtype=int)

# Combine legitimate and illegitimate labels
labels = np.concatenate([legitimate_labels, illegitimate_labels])

# Shuffle the labels
np.random.shuffle(labels)
labels = labels.flatten()

# Now you can use these labels with your data


In [6]:
#number_of_legitmate = 15000
# legitmate_data = data[0:number_of_legitmate]
# illegitmate_data = data[number_of_legitmate:]
#legitmate_labels = labels[0:number_of_legitmate]
#illegitmate_labels = labels[number_of_legitmate:]

# legitmate = 0
# illegitmate = 1
import numpy as np

# Create labels
#labels = np.array([0]*15000 + [1]*5000)
labels = np.array([0]*15000 + [1]*5000)
#labels = np.random.randint(0, 2, size=labels.shape)
#labels = labels.flatten()
#binary_labels = np.random.randint(0, 2, labels)
#binary_labels = np.where(labels <=15, 0, 1)

In [9]:
labels.shape

(10000,)

In [10]:
print(data.shape)
print(labels.shape)
print(np.unique(labels))

(10000, 8192)
(10000,)
[0 1]


In [11]:
#SOME PREPROCESSING
data_real_values = np.real(data).reshape(data.shape[0], 1, data.shape[1])
data_real_values = data_real_values/np.max(data_real_values)
data_real_values.shape
data_real_values_train, data_real_values_test, label_train, label_test = train_test_split(data_real_values, labels, test_size=0.2, shuffle=True)

# HYPERPAMETER TUNNING
# learning rate of optimiser, high epochs for low lr
# padding in layer 1 and corresponding maxpooling layers in cnn_model

# NOTES
# default adam with binary_crossentropy not good, but lr=0.0001
# 0.001 sgd with binary_crossentropy good
# default focalbinary_crossentropy with default adam good
# combimed cnn+lstm with 0.001 rms prop good 

In [13]:
print(np.where(label_test==1)[0].size)

592


In [14]:
print(data_real_values.shape)
print(data_real_values_train.shape)
print(data_real_values_test.shape)



(10000, 1, 8192)
(8000, 1, 8192)
(2000, 1, 8192)


In [ ]:
X=data_real_values
y=labels
X_train=data_real_values_train
X_val=X_test= data_real_values_test

y_train=label_train, 
y_val=y_test=label_test

In [ ]:
print(data_real_values.shape)
print(data_real_values_train.shape)
print(data_real_values_test.shape)



In [15]:
import tensorflow as tf
import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import  Conv1D, MaxPooling1D, Flatten, Dense
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, GRU, SimpleRNN

In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dropout, Flatten, Dense, BatchNormalization

cnn_model1 = Sequential()
cnn_model1.add(Conv1D(filters=32, kernel_size=3, activation='relu', padding='same', input_shape=(1,8192)))
cnn_model1.add(BatchNormalization())
cnn_model1.add(MaxPooling1D(pool_size=2, padding='same'))
cnn_model1.add(Dropout(0.3))
cnn_model1.add(Conv1D(64, kernel_size=3, activation='relu', padding='same'))
cnn_model1.add(BatchNormalization())
cnn_model1.add(MaxPooling1D(pool_size=2, padding='same'))
cnn_model1.add(Dropout(0.3))
cnn_model1.add(Conv1D(128, kernel_size=3, activation='relu', padding='same'))
cnn_model1.add(BatchNormalization())
cnn_model1.add(MaxPooling1D(pool_size=2, padding='same'))
cnn_model1.add(Dropout(0.3))
cnn_model1.add(Flatten())
cnn_model1.add(Dense(128, activation='relu'))
cnn_model1.add(Dropout(0.5))
cnn_model1.add(Dense(1, activation='sigmoid'))

#cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Model summary
#cnn_model.summary()


/home/codespace/.python/current/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:


cnn_model = Sequential()
cnn_model.add(Conv1D(filters=32, kernel_size=3, activation='relu', padding='same', input_shape=(1, 8192)))
#cnn_model.add(MaxPooling1D(pool_size=2, padding='same'))
#cnn_model.add(Dropout(0.2))
cnn_model.add(Conv1D(64, kernel_size=3, activation='relu', padding='same'))
cnn_model.add(Conv1D(128, kernel_size=3, activation='relu', padding='same'))
# cnn_model.add(Conv1D(256, kernel_size=3, activation='relu', padding='same'))
# cnn_model.add(Conv1D(512, kernel_size=3, activation='relu', padding='same'))
cnn_model.add(Flatten())
cnn_model.add(Dense(64, activation='relu'))
cnn_model.add(Dense(1, activation='sigmoid'))

#cnn_model.summary()

In [17]:
#Define the LSTM model
lstm_model = Sequential()
lstm_model.add(LSTM(64, activation='relu', input_shape=(1, 8192)))
lstm_model.add(Dense(128, activation='relu'))
lstm_model.add(Dropout(0.5))
lstm_model.add(Dense(1, activation='sigmoid'))

/home/codespace/.python/current/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
# Create a Sequential model
mlp_model = Sequential()
    
# Add layers to the model,
mlp_model.add(Dense(128, activation='relu', input_shape=(1,8192)))  # Input layer with 10 input features and 128 neurons,
Flatten()
mlp_model.add(Dense(64, activation='relu'))
mlp_model.add(Dropout(0.5))                # Hidden layer with 64 neurons and ReLU activation,
#mlp_model.add(Dense(128, activation='relu'))                # Another hidden layer with 32 neurons and ReLU activation\n",
mlp_model.add(Dense(1, activation='softmax'))              # Output layer with 1 neuron and sigmoid activation for binary classification\n",
   
#Print the model summary
#model.summary()"
   

In [12]:
simpleRNN_model = Sequential()
   
# Add SimpleRNN layer to the model
simpleRNN_model.add(SimpleRNN(64, input_shape=(1, 8192)))  # SimpleRNN layer with 64 units and input shape (10, 1)
   
# Add Dense layer to the model
simpleRNN_model.add(Dense(1, activation='sigmoid'))  # Output layer with sigmoid activation for binary classification
#model.summary()

In [ ]:
gru_model = Sequential()
 
# Add GRU layer to the model
gru_model.add(GRU(64, input_shape=(1, 8192)))  # GRU layer with 64 units and input shape (10, 1)

# Add Dense layer to the model\n",
gru_model.add(Dense(30, activation='sigmoid'))  # Output layer with sigmoid activation for binary classification"
  

In [ ]:
cnn_lstm_model = Sequential()
cnn_lstm_model.add(Conv1D(filters=32, kernel_size=3, activation='relu', padding='same', input_shape=(1, 8192)))
#cnn_lstm_model.add(MaxPooling1D(pool_size=2, padding='same'))
#cnn_lstm_model.add(Dropout(0.2))
cnn_lstm_model.add(Conv1D(64, kernel_size=1, activation='relu'))
cnn_lstm_model.add(Conv1D(64, kernel_size=1, activation='relu'))
# cnn_lstm_model.add(Flatten())
# cnn_lstm_model.add(Dense(64, activation='relu'))

cnn_lstm_model.add(LSTM(64, activation='relu'))
cnn_lstm_model.add(Dense(128, activation='relu'))
#cnn_lstm_model.add(Dropout(0.5))
cnn_lstm_model.add(Dense(1, activation='sigmoid'))

In [ ]:
cnn_lstm_model = Sequential()
cnn_lstm_model.add(Conv1D(filters=32, kernel_size=3, activation='relu', padding='same', input_shape=(1, 8192)))
cnn_lstm_model.add(MaxPooling1D(pool_size=2, padding='same'))
cnn_lstm_model.add(Dropout(0.2))
cnn_lstm_model.add(Conv1D(64, kernel_size=1, activation='relu'))
cnn_lstm_model.add(Conv1D(64, kernel_size=1, activation='relu'))
# cnn_lstm_model.add(Flatten())\n",
# cnn_lstm_model.add(Dense(64, activation='relu'))\n",
    
cnn_lstm_model.add(LSTM(64, activation='relu'))
cnn_lstm_model.add(Dense(128, activation='relu'))
cnn_lstm_model.add(Dropout(0.5))
#cnn_lstm_model.add(Dense(64, activation='relu'))                                                                                                                                         (1, activation='sigmoid'))\n",
   
cnn_lstm_model.add(Dense(30, activation='sigmoid'))

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Flatten
    
# Define input shape for 1D CNN\n",
#input_shape = (1, 8192)  # Sequence length of 100 with 1 feature\n",
    
# Create a Sequential model\n",
CNN_LSTM_model = Sequential()
    
# CNN layers\n",
CNN_LSTM_model.add(Conv1D(filters=64, kernel_size=3, activation='relu', padding ='same', input_shape=(1, 8192)))
CNN_LSTM_model.add(MaxPooling1D(pool_size=2, padding='same'))
CNN_LSTM_model.add(Conv1D(filters=128, kernel_size=1, activation='relu'))
CNN_LSTM_model.add(MaxPooling1D(pool_size=2, padding='same'))
CNN_LSTM_model.add(Dropout(0.2))

# LSTM layer
CNN_LSTM_model.add(LSTM(64, return_sequences=False))

CNN_LSTM_model.add(Dense(30, activation='sigmoid'))
    
# Compile the model\n",
#model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])\n",

# Print the model summary\n",
#model.summary()

In [19]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
#model_to_train = cnn_model
#model_to_train = cnn_model1
model_to_train = lstm_model
#model_to_train = CuDNNlstm_model\n",
#model_to_train = cnn_lstm_model
#model_to_train = mlp_model
#model_to_train = simpleRNN_model
#model_to_train = gru_model\n",
#model_to_train=CNN_LSTM_model\n",
    
loss1 = 'sparse_categorical_crossentropy'
#loss2 = 'binary_crossentropy'
#loss3 = 'binary_focal_crossentropy',
    
#optimizer = keras.optimizers.Adam(learning_rate=0.001) #https://keras.io/api/optimizers/adam/
#optimizer = keras.optimizers.RMSprop(learning_rate=0.001) # https://keras.io/api/optimizers/rmsprop/
#optimizer = keras.optimizers.SGD(learning_rate=0.001) #https://keras.io/api/optimizers/sgd/
    



callbacks = [
   EarlyStopping(monitor='val_loss',min_delta = 0, patience=20 ),
ReduceLROnPlateau(monitor='val_loss', min_delta = 0, factor=0.2, patience=5, min_lr=0.01,verbose=1)
]


#for model in models:
model_to_train.compile(optimizer='adam', loss=loss2, metrics=['accuracy'])
train_history = model_to_train.fit(data_real_values_train, label_train, 
                                       validation_data = (data_real_values_test, label_test),
                                       epochs=50, batch_size=32,
                                       callbacks=callbacks,
                                       )

   


Epoch 1/50


2024-05-29 21:22:49.853722: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 262144000 exceeds 10% of free system memory.


247/250 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6866 - loss: 0.6230

2024-05-29 21:22:53.307016: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 65536000 exceeds 10% of free system memory.


250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.6868 - loss: 0.6229 - val_accuracy: 0.7040 - val_loss: 0.6075 - learning_rate: 0.0010
Epoch 2/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.6908 - loss: 0.6190 - val_accuracy: 0.7040 - val_loss: 0.6075 - learning_rate: 0.0010
Epoch 3/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.6939 - loss: 0.6180 - val_accuracy: 0.7040 - val_loss: 0.6076 - learning_rate: 0.0010
Epoch 4/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.7052 - loss: 0.6084 - val_accuracy: 0.7040 - val_loss: 0.6160 - learning_rate: 0.0010
Epoch 5/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.7073 - loss: 0.6127 - val_accuracy: 0.7040 - val_loss: 0.6076 - learning_rate: 0.0010
Epoch 6/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.6920 - loss: 0.6182 - val_accuracy: 0.7040 - val_loss: 0.6076 - learning_rate: 0.0010
Epoch 7/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.6990 - loss: 0.6135 - val_a

In [ ]:
model_to_train.save('cnn_model1.h5')

In [ ]:
# load and evaluate a saved model\n",
from numpy import loadtxt
from tensorflow.keras.models import load_model

# load model\n",
model = load_model('cnn_model1.h5')
# summarize model.\n",
#model.summary()

# evaluate the model\n",
score = model.evaluate(data_real_values_test, label_test, verbose=0)
print("%s: %.2f%%" % (model.metrics_names[1], score[1]*100))
   

In [ ]:
import pandas as pd
pd.DataFrame(train_history.history).plot(figsize=(8,5))
plt.title(f'hey')
# plt.savefig('./Plots/Base_model/7.')
plt.show()

In [ ]:

# data_real_values_test, label_train, label_testa
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

all_labels = np.unique(binary_labels)
predictions = model_to_train.predict(data_real_values_test)
predictions = np.argmax(predictions, axis=1).reshape(predictions.shape[0], 1)
confusion_matrix = metrics.confusion_matrix(label_test, predictions)
cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix = confusion_matrix, display_labels = all_labels)
cm_display.plot()
plt.show()
precision_recall_fscore_support(label_test, predictions, average='macro')



In [ ]:
path_to_model = './trained_models/my_model.h5'

#model_to_train.save(path_to_model)

loaded_model = tf.keras.models.load_model(path_to_model)

#loaded_model.summary()

In [ ]:
file_path = "/workspaces/work/dataset_training_aug.h5"   
dev_range = np.arange(0,30, dtype = int), 
pkt_range = np.arange(0, 30000, dtype = int)

LoadDatasetObj = LoadDataset()
    
# Load preamble IQ samples and labels.
TEST_data, TEST_label = LoadDatasetObj.load_iq_samples(file_path, 
                                                 dev_range, 
                                                 pkt_range)

                                                

In [ ]:
#SOME PREPROCESSING
TEST_data_real_values = np.real(TEST_data).reshape(TEST_data.shape[0], 1, TEST_data.shape[1])


In [ ]:
# loss, acc = loaded_model.evaluate(TEST_data, TEST_label)

All_labels = np.unique(TEST_label)
TEST_predictions = loaded_model.predict(TEST_data_real_values)

TEST_predictions = np.argmax(TEST_predictions, axis=1).reshape(TEST_predictions.shape[0], 1)

TEST_confusion_matrix = metrics.confusion_matrix(TEST_label, TEST_predictions)
cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix = TEST_confusion_matrix, display_labels = All_labels)
cm_display.plot()
plt.show()

precision_recall_fscore_support(TEST_label, TEST_predictions, average='macro')

In [ ]:
#SOME PREPROCESSING
data_real_values = np.real(data).reshape(data.shape[0], data.shape[1])
data_real_values = data_real_values/np.max(data_real_values)
data_real_values.shape
data_real_values_train, data_real_values_test, label_train, label_test = train_test_split(data_real_values, label, test_size=0.2)

# HYPERPAMETER TUNNING
# learning rate of optimiser, high epochs for low lr
# padding in layer 1 and corresponding maxpooling layers in cnn_model

# NOTES
# default adam with binary_crossentropy not good, but lr=0.0001
# 0.001 sgd with binary_crossentropy good
# default focalbinary_crossentropy with default adam good
# combimed cnn+lstm with 0.001 rms prop good 

In [ ]:
data

In [ ]:
label

In [ ]:
print(data_real_values.shape)
print(data_real_values_train.shape)
print(data_real_values_test.shape)



In [ ]:
data_real_values.shape[1]

In [ ]:
input_shape=data_real_values.shape
print(input_shape)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_score
#confusion matrix

label_pred = model.predict(data_real_values_test)
label_pred_classes = np.argmax(label_pred, axis=1)
label_true = np.argmax(label_test, axis=1)

cm = confusion_matrix(label_true, label_pred_classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap=plt.cm.Blues)
plt.show()


In [ ]:
precision = precision_score(label_true, label_pred_classes, average='macro')
print(f'Precision: {precision:.2f}')

plt.bar(['Precision'], [precision])
plt.ylim(0, 1)
plt.title('Precision Score')
plt.show()


In [ ]:
import tensorflow as tf
import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import  Conv1D, MaxPooling1D, Flatten, Dense
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout


cnn_model = Sequential()
cnn_model.add(Conv1D(filters=32, kernel_size=3, activation='relu', padding='same', input_shape=(1, 8192)))
#cnn_model.add(MaxPooling1D(pool_size=2, padding='same'))
#cnn_model.add(Dropout(0.2))
cnn_model.add(Conv1D(64, kernel_size=1, activation='relu'))
cnn_model.add(Conv1D(64, kernel_size=1, activation='relu'))
cnn_model.add(Flatten())
cnn_model.add(Dense(64, activation='relu'))
cnn_model.add(Dense(1, activation='sigmoid'))

#cnn_model.summary()

In [ ]:
# Define the LSTM model
lstm_model = Sequential()
lstm_model.add(LSTM(64, activation='relu', input_shape=(1, 8192)))
lstm_model.add(Dense(128, activation='relu'))
lstm_model.add(Dropout(0.5))
lstm_model.add(Dense(1, activation='sigmoid'))



In [ ]:
cnn_lstm_model = Sequential()
cnn_lstm_model.add(Conv1D(filters=32, kernel_size=3, activation='relu', padding='same', input_shape=(1, 8192)))
#cnn_lstm_model.add(MaxPooling1D(pool_size=2, padding='same'))
#cnn_lstm_model.add(Dropout(0.2))
cnn_lstm_model.add(Conv1D(64, kernel_size=1, activation='relu'))
cnn_lstm_model.add(Conv1D(64, kernel_size=1, activation='relu'))
# cnn_lstm_model.add(Flatten())
# cnn_lstm_model.add(Dense(64, activation='relu'))

cnn_lstm_model.add(LSTM(64, activation='relu'))
cnn_lstm_model.add(Dense(128, activation='relu'))
#cnn_lstm_model.add(Dropout(0.5))
cnn_lstm_model.add(Dense(1, activation='sigmoid'))

In [ ]:
model_to_train = cnn_lstm_model

#loss2 = 'binary_crossentropy'
loss3 = keras.losses.BinaryFocalCrossentropy()

#optimizer = keras.optimizers.Adam(learning_rate=0.0001) #https://keras.io/api/optimizers/adam/
optimizer = keras.optimizers.RMSprop(learning_rate=0.001) # https://keras.io/api/optimizers/rmsprop/
#optimizer = keras.optimizers.SGD(learning_rate=0.001) #https://keras.io/api/optimizers/sgd/

model_to_train.compile(optimizer=optimizer, loss=loss3, metrics=['accuracy'])

train_history = model_to_train.fit(data_real_values_train, label_train, 
                                   validation_data = (data_real_values_test, label_test),
                                   epochs=100, batch_size=32)

In [ ]:
import pandas as pd
pd.DataFrame(train_history.history).plot(figsize=(8,5))
plt.title(f'hey')
# plt.savefig('./Plots/Base_model/7.')
plt.show()

In [ ]:
# data_real_values_test, label_train, label_testa
all_labels = np.unique(label)
predictions = model_to_train.predict(data_real_values_test)
confusion_matrix = metrics.confusion_matrix(label_test, predictions)
precision_recall_fscore_support(label_test, predictions, average='macro')
cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix = confusion_matrix, display_labels = all_labels)
cm_display.plot()
plt.show()